# Módulo 44: Preparación de datos

Autor: Juliho Castillo Colmenares

> Lección complementaria del ejercicio de este módulo: [enunciado](m44-preparacion-de-datos-guia.ipynb) y [solución](m44-preparacion-de-datos-solucion.ipynb). No es una tarea entregada del programa.

Un dashboard solo es tan bueno como los datos que lo alimentan. Antes de dibujar un solo gráfico hay que leer los archivos, darles la forma correcta y limpiar lo que viene mal. En esta sección esa tarea la hacen **pandas** y **DuckDB**, y en esta lección la recorremos con los datos de ventas de la práctica (`../practica-ventas/datos-ventas/`), que traen casi todos los problemas típicos de un extracto real.

1. Del archivo al dashboard
2. Leer CSV, TXT, una carpeta y Excel
3. La limpieza como receta: funciones encadenadas
4. Perfilado de columnas
5. Limpieza de los casos reales
6. Anexar frente a combinar
7. Columnas calculadas
8. Guardar el modelo
9. Errores frecuentes
10. Ejercicio

## 1. Del archivo al dashboard

Construir un tablero tiene tres etapas, y en esta sección cada una tiene su herramienta:

| Etapa | Herramienta | Pregunta que responde | Módulos |
| --- | --- | --- | --- |
| Preparación | pandas (y DuckDB) | ¿Los datos están limpios y con la forma correcta? | 44 |
| Modelado y medidas | DuckDB y SQL | ¿Cómo se relacionan las tablas y qué se calcula? | 45 y 47 |
| Visualización | Plotly y Streamlit | ¿Cómo se comunica el resultado? | 46, 48, 49 y 50 |

La regla de oro de la preparación es **no modificar los archivos de origen**. La limpieza se escribe como código: una receta que se vuelve a ejecutar cada vez que llega un extracto nuevo. Si el mes que viene llega otro CSV con el mismo formato, basta con volver a correr la receta.

Las herramientas de BI con interfaz gráfica (como Power Query) hacen lo mismo registrando "pasos aplicados"; aquí cada paso es una línea de pandas, que además se puede revisar, versionar en git y probar.

In [1]:
from pathlib import Path

import duckdb
import pandas as pd

DATOS = Path("../practica-ventas/datos-ventas")
pd.set_option("display.max_columns", 20)

sorted(p.relative_to(DATOS).as_posix() for p in DATOS.rglob("*") if p.is_file())

['catalogo.xlsx',
 'paises.txt',
 'ventas-2019.csv',
 'ventas-por-anio/ventas-2020.csv',
 'ventas-por-anio/ventas-2021.csv']

## 2. Leer CSV, TXT, una carpeta y Excel

Antes de llamar a `read_csv` conviene mirar los **bytes** de las primeras líneas. Así se descubren problemas que pandas no avisa: marcas invisibles al inicio, separadores raros o comas sobrantes.

In [2]:
def primeras_lineas(ruta, n=3):
    # Muestra las primeras líneas tal como están en disco, sin interpretar
    with open(ruta, "rb") as archivo:
        for _ in range(n):
            print(archivo.readline())


primeras_lineas(DATOS / "ventas-2019.csv")
print()
primeras_lineas(DATOS / "ventas-por-anio" / "ventas-2021.csv")
print()
primeras_lineas(DATOS / "paises.txt", 4)

b'\xef\xbb\xbfID_T,ID_Country,Date,ID_Store,ID_Client,ID_Product,Quantity,Price\r\n'
b'1,9,10/24/2019,1,551,291,155,233\r\n'
b'2,2,6/17/2019,153,774,710,25,415\r\n'

b'\xef\xbb\xbfID_T,ID_Country,Date,ID_Store,ID_Client,ID_Product,Quantity,Price,\r\n'
b'1,1,12/5/2021,102,94,317,175,326,\r\n'
b'2,8,8/26/2021,16,30,951,33,366,\r\n'

b'1|US\r\n'
b'1|US\r\n'
b'2|CANADA\r\n'
b'3|MEXICO\r\n'


Tres hallazgos:

- `ventas-2019.csv` empieza con `\xef\xbb\xbf`, el **BOM** (*byte order mark*) de UTF-8. Si se lee con `encoding="utf-8"`, la primera columna se llama `"\ufeffID_T"` en lugar de `ID_T`. Con `encoding="utf-8-sig"` el BOM se descarta.
- `ventas-2021.csv` termina **cada línea en coma**, encabezado incluido: pandas verá una novena columna vacía.
- `paises.txt` usa `|` como separador y **no tiene encabezado**.

Cada formato tiene su lector:

| Archivo | Lector | Parámetros clave |
| --- | --- | --- |
| `ventas-2019.csv` | `pd.read_csv` | `encoding="utf-8-sig"` |
| `ventas-por-anio/` | `read_csv` en un bucle, o `duckdb.read_csv` con un patrón `*.csv` | `usecols=range(8)` para la coma final |
| `paises.txt` | `pd.read_csv` | `sep="\|"`, `header=None`, `names=[...]` |
| `catalogo.xlsx` | `pd.read_excel` | `sheet_name`, `header` |

In [3]:
ventas_2019 = pd.read_csv(DATOS / "ventas-2019.csv", encoding="utf-8-sig")
print(ventas_2019.shape)
ventas_2019.head(3)

(300000, 8)


,ID_T,ID_Country,Date,ID_Store,ID_Client,ID_Product,Quantity,Price
0,1,9,10/24/2019,1,551,291,155,233
1,2,2,6/17/2019,153,774,710,25,415
2,3,10,3/4/2019,159,356,962,445,58


In [4]:
paises_crudo = pd.read_csv(DATOS / "paises.txt", sep="|", header=None, names=["id_pais", "pais"])
paises_crudo

,id_pais,pais
0,1,US
1,1,US
2,2,CANADA
3,3,MEXICO
4,4,BRAZIL
5,5,SPAIN
6,6,COLOMBIA
7,7,CHILE
8,8,PERU
9,9,PORTUGAL


`read_excel` con `sheet_name=None` lee **todas** las hojas y devuelve un diccionario. Con `header=None` vemos las hojas tal cual, sin que pandas adivine dónde está el encabezado.

In [5]:
hojas = pd.read_excel(DATOS / "catalogo.xlsx", sheet_name=None, header=None)
{nombre: hoja.shape for nombre, hoja in hojas.items()}

{'t_customer': (956, 4), 't_store': (207, 5), 't_product': (1018, 5)}

In [6]:
hojas["t_store"].head(5)

,0,1,2,3,4
0,As of Date:2020,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN
2,ID_Store,Name_Store,Addres,Format,Status
3,2,Huels-Robel,1029 Petterle Trail,AAA,Open
4,4,Smith-Lowe,8 Lake View Trail,AAA,Open


**Una carpeta de archivos.** Cuando los datos llegan en varios archivos con la misma estructura, se leen todos con un patrón y se apilan. En pandas basta con `glob` y una lista por comprensión; guardar el nombre del archivo en una columna ayuda a rastrear de dónde viene cada fila.

In [7]:
archivos = sorted((DATOS / "ventas-por-anio").glob("*.csv"))
partes = [
    pd.read_csv(archivo, encoding="utf-8-sig", usecols=range(8)).assign(archivo=archivo.name)
    for archivo in archivos
]
ventas_carpeta = pd.concat(partes, ignore_index=True)
ventas_carpeta.groupby("archivo").size()

archivo
ventas-2020.csv    499000
ventas-2021.csv    700000
dtype: int64

DuckDB hace lo mismo en una sola consulta SQL: `read_csv` acepta un patrón con `*` y la opción `filename = true` agrega la columna con el origen de cada fila. Es útil cuando los archivos son muchos o muy grandes, porque DuckDB los lee en paralelo sin cargarlos antes en memoria.

In [8]:
patron = (DATOS / "ventas-por-anio" / "*.csv").as_posix()
duckdb.sql(f"""
    SELECT parse_filename(filename) AS archivo, count(*) AS filas, min(Date) AS primera, max(Date) AS ultima
    FROM read_csv('{patron}', filename = true, union_by_name = true, dateformat = '%m/%d/%Y')
    GROUP BY archivo
    ORDER BY archivo
""").df()

,archivo,filas,primera,ultima
0,ventas-2020.csv,499000,2020-01-01,2020-12-31
1,ventas-2021.csv,700000,2021-01-01,2021-12-31


## 3. La limpieza como receta: funciones encadenadas

Una buena receta de limpieza se lee de arriba abajo, paso a paso. En pandas eso se logra con funciones pequeñas encadenadas con `.pipe`: cada función recibe la tabla del paso anterior y devuelve una tabla nueva, sin modificar la original.

In [9]:
def leer_ventas_csv(ruta):
    # utf-8-sig quita el BOM; usecols=range(8) descarta la columna de la coma final
    return pd.read_csv(ruta, encoding="utf-8-sig", usecols=range(8))


def tipar_fechas(ventas):
    return ventas.assign(Date=pd.to_datetime(ventas["Date"], format="%m/%d/%Y"))


def quitar_id_t(ventas):
    # ID_T vuelve a empezar en 1 en cada archivo: no sirve como clave
    return ventas.drop(columns="ID_T")


ventas_2019 = (
    leer_ventas_csv(DATOS / "ventas-2019.csv")
    .pipe(tipar_fechas)
    .pipe(quitar_id_t)
)
ventas_2019.dtypes

ID_Country             int64
Date          datetime64[ns]
ID_Store               int64
ID_Client              int64
ID_Product             int64
Quantity               int64
Price                  int64
dtype: object

Las operaciones de limpieza más usadas son pocas. Esta tabla las relaciona con los botones de una herramienta de BI con interfaz gráfica:

| pandas | Equivalente en una herramienta de BI | Qué hace |
| --- | --- | --- |
| `read_csv(header=...)`, `names=` | Usar la primera fila como encabezado | Define los nombres de columna |
| `astype`, `to_datetime(format=...)` | Tipo de datos | Asigna tipos (entero, decimal, fecha, texto) |
| `str.split(..., expand=True)` | Dividir columna por delimitador | Separa un texto en varias columnas |
| `read_excel(header=n)`, filtros | Quitar filas superiores | Descarta filas basura |
| `drop_duplicates` | Quitar duplicados | Elimina filas repetidas |
| `pd.concat` | Anexar consultas | Apila tablas con las mismas columnas |
| `merge` | Combinar consultas | Agrega columnas por una clave |

## 4. Perfilado de columnas

Antes de limpiar hay que **medir**. Una función de perfil que diga, para cada columna, cuántos valores son válidos, cuántos distintos y cuántos únicos delata casi todos los problemas de un vistazo.

In [10]:
def perfil(tabla):
    # Calidad y distribución de cada columna
    return pd.DataFrame({
        "tipo": tabla.dtypes.astype(str),
        "validos_%": tabla.notna().mean().mul(100).round(1),
        "vacios": tabla.isna().sum(),
        "distintos": tabla.nunique(),
        "unicos": tabla.apply(lambda columna: (columna.value_counts() == 1).sum()),
    })


tiendas_crudo = pd.read_excel(DATOS / "catalogo.xlsx", sheet_name="t_store", header=2)
print(len(tiendas_crudo), "filas")
perfil(tiendas_crudo)

204 filas


,tipo,validos_%,vacios,distintos,unicos
ID_Store,int64,100.0,0,200,196
Name_Store,object,100.0,0,199,195
Addres,object,100.0,0,200,196
Format,object,100.0,0,3,0
Status,object,100.0,0,2,0


El perfil de `t_store` muestra dos problemas:

- `ID_Store` tiene 204 filas pero solo 200 valores distintos: hay **tiendas repetidas**.
- `Format` tiene tres valores distintos, pero se esperaba `A`, `AA` y `AAA`. Con `repr` se ven los espacios y las minúsculas:

In [11]:
print(tiendas_crudo["Format"].map(repr).value_counts(), end="\n\n")
tiendas_crudo[tiendas_crudo["ID_Store"].duplicated(keep=False)].sort_values("ID_Store")

Format
'aa'      71
'AAA'     70
'   A'    63
Name: count, dtype: int64



,ID_Store,Name_Store,Addres,Format,Status
5,17,Rippin LLC,30 Badeau Parkway,AAA,Open
6,17,Rippin LLC,30 Badeau Parkway,AAA,Closed
188,167,Treutel-Ullrich,36 Summer Ridge Center,A,Open
189,167,Treutel-Ullrich,36 Summer Ridge Center,A,Closed
199,188,Schoen-Turcotte,34 Pennsylvania Center,A,Open
200,188,Schoen-Turcotte,34 Pennsylvania Center,A,Closed
202,197,Schumm Inc,0 Killdeer Circle,A,Closed
203,197,Schumm Inc,0 Killdeer Circle,A,Open


## 5. Limpieza de los casos reales

Los archivos de `datos-ventas` traen cinco problemas distintos:

| Origen | Problema | Solución en pandas |
| --- | --- | --- |
| `paises.txt` | Sin encabezado; `1\|US`, `4\|BRAZIL` y `5\|SPAIN` aparecen dos veces (13 filas, 10 países) | `names=[...]` y `drop_duplicates` |
| `ventas-2021.csv` | Cada línea termina en coma, lo que crea una novena columna vacía | `usecols=range(8)` |
| `t_store` | Dos filas basura antes del encabezado; `Format` escrito como `"aa"`, `"AAA"` y `"   A"`; las tiendas 17, 167, 188 y 197 aparecen como `Open` y como `Closed` | `header=2`, `str.strip().str.upper()` y filtrar `Status == "Open"` |
| `t_customer` | `ID_Client` empaqueta cinco datos; al final hay filas vacías y un pie `CRM Extract by Data team 862992`; `zone` vale `?` en todas las filas | Filtrar por contenido, `str.split("\|", expand=True)` y descartar columnas vacías |
| `t_product` | `Product ID` tiene la forma `ID-1000\|Product`; 17 filas inactivas (`Active = 0`) repiten 15 productos: 13 aparecen dos veces y 2, tres veces | `str.extract` y filtrar `Active == 1` |

**Países.**

In [12]:
paises = paises_crudo.drop_duplicates().sort_values("id_pais", ignore_index=True)
print(len(paises_crudo), "filas ->", len(paises), "países; clave única:", paises["id_pais"].is_unique)
paises

13 filas -> 10 países; clave única: True


,id_pais,pais
0,1,US
1,2,CANADA
2,3,MEXICO
3,4,BRAZIL
4,5,SPAIN
5,6,COLOMBIA
6,7,CHILE
7,8,PERU
8,9,PORTUGAL
9,10,FRANCE


**Clientes.** El final de la hoja tiene filas vacías y un pie de página:

In [13]:
clientes_crudo = pd.read_excel(DATOS / "catalogo.xlsx", sheet_name="t_customer", dtype=str)
clientes_crudo.tail(5)

,ID_Client,first_name,last_name,zone
950,NaN,NaN,NaN,NaN
951,NaN,NaN,NaN,NaN
952,NaN,NaN,NaN,NaN
953,NaN,NaN,NaN,NaN
954,CRM Extract by Data team 862992,NaN,NaN,NaN


Filtrar las filas que empiezan por `ID|` elimina de una vez las vacías y el pie (`na=False` trata las vacías como "no empieza"). Después se divide el campo empaquetado por `|` en sus cinco partes:

In [14]:
validas = clientes_crudo[clientes_crudo["ID_Client"].str.startswith("ID|", na=False)]
partes = validas["ID_Client"].str.split("|", expand=True)
partes.head(3)

,0,1,2,3,4
0,ID,658,Hamlin/Zohrer,mail:hzohrerih@over-blog.com,+55-890-712-8982
1,ID,261,Karie/Zellmer,mail:kzellmer7g@gov.uk,+55-119-801-5241
2,ID,239,Tucky/Zapata,mail:tzapata6u@noaa.gov,+62-353-421-3814


In [15]:
nombre = partes[2].str.split("/", n=1, expand=True)
clientes = pd.DataFrame({
    "id_cliente": partes[1].astype(int),
    "nombre": nombre[0],
    "apellido": nombre[1],
    "correo": partes[3].str.removeprefix("mail:"),
    "telefono": partes[4],
}).sort_values("id_cliente", ignore_index=True)

print(len(clientes), "clientes; clave única:", clientes["id_cliente"].is_unique)
clientes.head(3)

950 clientes; clave única: True


,id_cliente,nombre,apellido,correo,telefono
0,1,Melany,Morrad,mmorrad0@walmart.com,+63-253-635-2782
1,2,Cally,Maginot,cmaginot1@google.ca,+86-574-743-6664
2,3,Elwyn,Timmes,etimmes2@google.de,+46-846-858-9359


**Productos.** La clave numérica está entre `ID-` y `|`. Una expresión regular con un grupo de captura la extrae sin dividir en columnas. Antes de filtrar se ve el problema de los duplicados:

In [16]:
productos_crudo = pd.read_excel(DATOS / "catalogo.xlsx", sheet_name="t_product")
productos_crudo["id_producto"] = productos_crudo["Product ID"].str.extract(r"ID-(\d+)\|", expand=False).astype(int)

repetidos = productos_crudo[productos_crudo["id_producto"].duplicated(keep=False)]
print(repetidos["id_producto"].nunique(), "productos repetidos")
repetidos.sort_values(["id_producto", "Active"]).head(4)

15 productos repetidos


,Product ID,Category,Sub-Category,Product Name,Active,id_producto
1008,ID-9|Product,Office Supplies,Binders,Fellowes PB500 Electric Punch Plastic Comb Bin...,0,9
1007,ID-9|Product,Office Supplies,Binders,Fellowes PB500 Electric Punch Plastic Comb Bin...,1,9
973,ID-43|Product,Office Supplies,Binders,Ibico EPK-21 Electric Binding System Out,0,43
972,ID-43|Product,Office Supplies,Binders,Ibico EPK-21 Electric Binding System,1,43


In [17]:
activos = productos_crudo[productos_crudo["Active"] == 1]
productos = (
    activos.rename(columns={"Category": "categoria", "Sub-Category": "subcategoria", "Product Name": "producto"})
    [["id_producto", "categoria", "subcategoria", "producto"]]
    .sort_values("id_producto", ignore_index=True)
)
print(len(productos), "productos, del", productos["id_producto"].min(), "al", productos["id_producto"].max(),
      "; clave única:", productos["id_producto"].is_unique)

1000 productos, del 1 al 1000 ; clave única: True


Quedan 1000 productos con `id_producto` del 1 al 1000, justo el rango de `ID_Product` en las ventas. La hoja de tiendas se limpia en el ejercicio de la sección 10.

## 6. Anexar frente a combinar

Hay dos formas de juntar tablas, y es fácil confundirlas:

| Operación | pandas | Dirección | SQL | Uso típico |
| --- | --- | --- | --- | --- |
| **Anexar** (append) | `pd.concat` | Apila filas | `UNION ALL` | Ventas de 2019 + ventas de 2020–2021 |
| **Combinar** (merge) | `merge` | Agrega columnas por una clave | `JOIN` | Agregar el nombre del país a cada venta |

Anexar exige que las tablas tengan **las mismas columnas con los mismos nombres**; si una columna falta en una tabla, pandas la rellena con `NaN`.

In [18]:
ventas_2020_2021 = ventas_carpeta.drop(columns="archivo").pipe(tipar_fechas).pipe(quitar_id_t)
ventas = pd.concat([ventas_2019, ventas_2020_2021], ignore_index=True)

print(f"{len(ventas_2019):,} + {len(ventas_2020_2021):,} = {len(ventas):,} filas".replace(",", " "))
print(f"Del {ventas['Date'].min():%Y-%m-%d} al {ventas['Date'].max():%Y-%m-%d}")

300 000 + 1 199 000 = 1 499 000 filas
Del 2019-01-01 al 2021-12-31


Desde aquí las columnas de la tabla de hechos usan nombres en español y minúsculas, los mismos que tendrá el modelo en el resto de la sección:

| Original | Modelo |
| --- | --- |
| `Date` | `fecha` |
| `ID_Country`, `ID_Store`, `ID_Client`, `ID_Product` | `id_pais`, `id_tienda`, `id_cliente`, `id_producto` |
| `Quantity`, `Price` | `cantidad`, `precio` |

In [19]:
ventas = ventas.rename(columns={
    "Date": "fecha", "ID_Country": "id_pais", "ID_Store": "id_tienda", "ID_Client": "id_cliente",
    "ID_Product": "id_producto", "Quantity": "cantidad", "Price": "precio",
})[["fecha", "id_pais", "id_tienda", "id_cliente", "id_producto", "cantidad", "precio"]]
ventas.head(3)

,fecha,id_pais,id_tienda,id_cliente,id_producto,cantidad,precio
0,2019-10-24,9,1,551,291,155,233
1,2019-06-17,2,153,774,710,25,415
2,2019-03-04,10,159,356,962,445,58


Combinar, en cambio, agrega columnas. `validate="many_to_one"` hace que pandas compruebe que la clave es única en la tabla de la derecha; si no lo fuera, el `merge` multiplicaría filas en silencio.

In [20]:
con_pais = ventas.merge(paises, on="id_pais", how="left", validate="many_to_one")


def megabytes(tabla):
    return tabla.memory_usage(deep=True).sum() / 1e6


print(f"ventas: {megabytes(ventas):.0f} MB | con el nombre del país en cada fila: {megabytes(con_pais):.0f} MB")

ventas: 84 MB | con el nombre del país en cada fila: 178 MB


Pegar el nombre del país a 1,5 millones de filas repite diez textos 1,5 millones de veces. Por eso en un modelo de BI las dimensiones se guardan **por separado** y se unen solo al consultar (Módulo 45). DuckDB lo hace con SQL directamente sobre los DataFrames de pandas, sin copiarlos:

In [21]:
duckdb.sql("""
    SELECT p.pais, count(*) AS transacciones
    FROM ventas v
    JOIN paises p USING (id_pais)
    GROUP BY p.pais
    ORDER BY transacciones DESC
""").df()

,pais,transacciones
0,FRANCE,150550
1,US,150429
2,PORTUGAL,150254
3,BRAZIL,150188
4,CHILE,150039
5,COLOMBIA,149757
6,MEXICO,149720
7,SPAIN,149676
8,CANADA,149398
9,PERU,148989


## 7. Columnas calculadas

Una **columna calculada** se evalúa fila a fila. El importe de cada venta es cantidad por precio:

In [22]:
ventas["importe"] = ventas["cantidad"] * ventas["precio"]
print(ventas.dtypes, end="\n\n")
print(f"Ventas totales: {ventas['importe'].sum():,}".replace(",", " "))
ventas.head(3)

fecha          datetime64[ns]
id_pais                 int64
id_tienda               int64
id_cliente              int64
id_producto             int64
cantidad                int64
precio                  int64
importe                 int64
dtype: object

Ventas totales: 41 940 958 825


,fecha,id_pais,id_tienda,id_cliente,id_producto,cantidad,precio,importe
0,2019-10-24,9,1,551,291,155,233,36115
1,2019-06-17,2,153,774,710,25,415,10375
2,2019-03-04,10,159,356,962,445,58,25810


Las tres columnas son enteras (`int64`), así que la suma es exacta: 41 940 958 825. Si el precio tuviera centavos, convendría guardarlo en centavos enteros o con `Decimal` para no acumular errores de redondeo de punto flotante.

## 8. Guardar el modelo

La receta termina guardando las tablas limpias en un formato eficiente. **Parquet** es un formato columnar y comprimido: guarda los tipos (las fechas siguen siendo fechas), ocupa mucho menos que el CSV y se lee en una fracción del tiempo. Para no ensuciar el repositorio, aquí se escribe en una carpeta temporal:

In [23]:
import tempfile

csv_mb = sum(p.stat().st_size for p in DATOS.rglob("*.csv")) / 1e6
with tempfile.TemporaryDirectory() as carpeta:
    destino = Path(carpeta) / "ventas.parquet"
    ventas.to_parquet(destino, index=False, compression="zstd")
    parquet_mb = destino.stat().st_size / 1e6
    releida = pd.read_parquet(destino)

print(f"CSV: {csv_mb:.1f} MB | Parquet: {parquet_mb:.1f} MB")
print("Tipos conservados:", releida.dtypes.equals(ventas.dtypes))

CSV: 57.0 MB | Parquet: 14.6 MB
Tipos conservados: True


Toda esta receta está reunida en [`../practica-ventas/ventas.py`](../practica-ventas/ventas.py), con una función por tabla (`leer_ventas`, `leer_paises`, `leer_clientes`, `leer_productos`, `leer_tiendas`). El script [`preparar_datos.py`](../practica-ventas/preparar_datos.py) la ejecuta y guarda el resultado en `../practica-ventas/datos-modelo/`, de donde leen los módulos siguientes y las apps de Streamlit. La comprobación final confirma que la receta de esta lección produce exactamente las mismas tablas:

In [24]:
import sys

sys.path.insert(0, str(Path("../practica-ventas").resolve()))
import ventas as practica

pd.testing.assert_frame_equal(ventas, practica.leer_ventas())
pd.testing.assert_frame_equal(paises, practica.leer_paises())
pd.testing.assert_frame_equal(clientes, practica.leer_clientes())
pd.testing.assert_frame_equal(productos, practica.leer_productos())
print("La lección y ventas.py producen las mismas tablas.")

La lección y ventas.py producen las mismas tablas.


## 9. Errores frecuentes

**Fechas con el orden equivocado.** `10/24/2019` solo puede ser mes/día/año, pero `1/2/2019` es ambiguo. Sin formato explícito, pandas adivina, y con `dayfirst=True` lo interpreta como 1 de febrero sin avisar:

In [25]:
print(pd.to_datetime(pd.Series(["1/2/2019"]), dayfirst=True)[0].date(), "<- 1 de febrero: incorrecto")
print(pd.to_datetime(pd.Series(["1/2/2019"]), format="%m/%d/%Y")[0].date(), "<- 2 de enero: correcto")
try:
    pd.to_datetime(pd.Series(["10/24/2019"]), format="%d/%m/%Y")
except ValueError as error:
    print("Con el formato equivocado, al menos falla:", str(error).splitlines()[0])

2019-02-01 <- 1 de febrero: incorrecto
2019-01-02 <- 2 de enero: correcto
Con el formato equivocado, al menos falla: time data "10/24/2019" doesn't match format "%d/%m/%Y", at position 0. You might want to try:


**Leer sin saltar las filas basura.** Si `t_store` se lee sin `header=2`, las dos filas basura se convierten en encabezado y datos, y todas las columnas quedan como texto (`object`), incluida la clave:

In [26]:
pd.read_excel(DATOS / "catalogo.xlsx", sheet_name="t_store").dtypes

As of Date:2020    object
Unnamed: 1         object
Unnamed: 2         object
Unnamed: 3         object
Unnamed: 4         object
dtype: object

**La columna fantasma de la coma final.** Sin `usecols=range(8)`, `ventas-2021.csv` trae una novena columna vacía que, al anexarla con 2019, llena de `NaN` las filas de los otros años:

In [27]:
pd.read_csv(DATOS / "ventas-por-anio" / "ventas-2021.csv", encoding="utf-8-sig", nrows=3)

,ID_T,ID_Country,Date,ID_Store,ID_Client,ID_Product,Quantity,Price,Unnamed: 8
0,1,1,12/5/2021,102,94,317,175,326,NaN
1,2,8,8/26/2021,16,30,951,33,366,NaN
2,3,9,10/7/2021,38,83,644,141,53,NaN


**Tratar `ID_T` como clave única.** Cada archivo numera sus transacciones desde 1; después de anexar, el número 1 aparece tres veces. Si se necesita un identificador único, se crea después de anexar (por ejemplo, con `reset_index`).

In [28]:
ids = pd.concat(leer_ventas_csv(archivo)["ID_T"] for archivo in [DATOS / "ventas-2019.csv", *archivos])
print("Veces que aparece ID_T = 1:", (ids == 1).sum())

Veces que aparece ID_T = 1: 3


**Quitar filas por posición.** `clientes_crudo.iloc[:-5]` funciona hoy en `t_customer`, pero si el próximo extracto trae una fila vacía menos, borrará un cliente. Filtrar por contenido (`str.startswith("ID|")`) es más robusto.

**Normalizar texto sin quitar espacios.** `"   A".upper()` sigue siendo `"   A"`, distinto de `"A"`. Primero se recortan los espacios y luego se cambian las mayúsculas:

In [29]:
formatos = pd.Series(["aa", "AAA", "   A"])
print("Solo upper:      ", formatos.str.upper().map(repr).tolist())
print("strip y luego upper:", formatos.str.strip().str.upper().map(repr).tolist())

Solo upper:       ["'AA'", "'AAA'", "'   A'"]
strip y luego upper: ["'AA'", "'AAA'", "'A'"]


## 10. Ejercicio

La hoja `t_store` de `catalogo.xlsx` empieza así:

| 0 | 1 | 2 | 3 | 4 |
| --- | --- | --- | --- | --- |
| As of Date:2020 | NaN | NaN | NaN | NaN |
| NaN | NaN | NaN | NaN | NaN |
| ID_Store | Name_Store | Addres | Format | Status |
| 2 | Huels-Robel | 1029 Petterle Trail | AAA | Open |

1. Lee la hoja saltando las dos filas basura.
2. Renombra las columnas a `id_tienda`, `tienda`, `direccion`, `formato` y `estado`, con `id_tienda` entero.
3. Normaliza `formato` para que solo existan los valores `A`, `AA` y `AAA`.
4. Deja una sola fila por tienda, quedándote con la versión `Open`.
5. Agrega una columna `formato_orden` con 1 para `A`, 2 para `AA` y 3 para `AAA`, útil para ordenar gráficos.
6. Comprueba con el perfil de columnas que `id_tienda` tiene tantos valores distintos como filas.

### Solución

In [30]:
tiendas = (
    pd.read_excel(DATOS / "catalogo.xlsx", sheet_name="t_store", header=2)
    .rename(columns={"ID_Store": "id_tienda", "Name_Store": "tienda", "Addres": "direccion",
                     "Format": "formato", "Status": "estado"})
    .astype({"id_tienda": int})
    .assign(formato=lambda t: t["formato"].str.strip().str.upper())
    .query("estado == 'Open'")
    .assign(formato_orden=lambda t: t["formato"].map({"A": 1, "AA": 2, "AAA": 3}))
    .sort_values("id_tienda", ignore_index=True)
)
print(tiendas["formato"].value_counts().to_dict())
perfil(tiendas)

{'AA': 71, 'AAA': 69, 'A': 60}


,tipo,validos_%,vacios,distintos,unicos
id_tienda,int64,100.0,0,200,200
tienda,object,100.0,0,199,198
direccion,object,100.0,0,200,200
formato,object,100.0,0,3,0
estado,object,100.0,0,1,0
formato_orden,int64,100.0,0,3,0


Quedan 200 filas, una por tienda, con `id_tienda` del 1 al 200 (200 distintos, 200 únicos) y tres formatos. Si se quisiera conservar el historial de estados, la alternativa sería no filtrar y guardar en otra tabla el estado de cada tienda por fecha; para usar `tiendas` como dimensión, la clave debe ser única.

## Resumen

- La preparación se escribe como una receta de código que se repite con cada extracto, sin tocar los archivos de origen.
- Mirar los bytes del archivo antes de leerlo descubre el BOM, los separadores y las comas sobrantes; `encoding="utf-8-sig"`, `sep`, `header`, `names` y `usecols` los resuelven.
- Una función de perfil (válidos, distintos, únicos) revela duplicados, vacíos y categorías mal escritas antes de que lleguen al tablero.
- `pd.concat` anexa filas de tablas con las mismas columnas; `merge` combina columnas por una clave y conviene validarlo con `validate`.
- El resultado se guarda en Parquet, que conserva los tipos y ocupa una fracción del CSV. La receta completa vive en `ventas.py`.

En el **Módulo 45** se organizan estas tablas en un modelo de datos con DuckDB: tabla de hechos, dimensiones y relaciones.